# Case Study

Smart Financial Market
Analytics and Stock Prediction System using
Apache Spark

Q1. Spark Initialization and Data Loading

In [19]:
pip install pyspark

In [20]:
import os
from pyspark.sql import SparkSession

In [21]:
import os
from pyspark.sql import SparkSession
# Step 1: Initialize the Spark Session
# This turns on the Spark engine on your notebook backend.
spark = SparkSession.builder.appName("SimpleFinancialAnalytics") \
    .getOrCreate()

In [22]:
# Get the Spark Context (used for lower-level RDD operations later)
sc = spark.sparkContext
print("Spark Session successfully started!")


Spark Session successfully started!


In [23]:
# Step 2: Define paths to your files in Colab
stock_path = "/content/stock_market_data.csv"
trading_path = "/content/historical_trading.csv"
news_path = "/content/financial_news.csv"

In [24]:
# Step 3: Load the datasets into DataFrames
# 'header=True' means use the first row as column names
# 'inferSchema=True' tells Spark to automatically guess data types (numbers vs text)
df_stock = spark.read.csv(stock_path, header=True, inferSchema=True)
df_trading = spark.read.csv(trading_path, header=True, inferSchema=True)
df_news = spark.read.csv(news_path, header=True, inferSchema=True)

In [25]:
# Step 4: Display the structural blueprint (Schema) and sample records
print("STOCK MARKET DATA SCHEMA")
df_stock.printSchema()
print("Sample Records:")
df_stock.show(3)

STOCK MARKET DATA SCHEMA
root
 |-- Date: date (nullable = true)
 |-- Stock_ID: string (nullable = true)
 |-- Sector: string (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: integer (nullable = true)
 |-- Sentiment_Score: double (nullable = true)
 |-- Target: integer (nullable = true)

Sample Records:
+----------+--------+------+------+------------------+------+------+------+---------------+------+
|      Date|Stock_ID|Sector|  Open|              High|   Low| Close|Volume|Sentiment_Score|Target|
+----------+--------+------+------+------------------+------+------+------+---------------+------+
|2024-01-01|     TCS|    IT|257.76|            259.09|252.41|255.41| 84105|           0.89|     0|
|2024-01-01|    INFY|    IT|242.85|251.91000000000003|239.81|250.11| 44593|          -0.03|     1|
|2024-01-01|RELIANCE|Energy|314.94|            315.14|309.96|312.51| 38794|  

In [26]:
print("HISTORICAL TRADING SCHEMA")
df_trading.printSchema()
print("Sample Records:")
df_trading.show(3)


HISTORICAL TRADING SCHEMA
root
 |-- Trade_ID: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- Stock_ID: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price: double (nullable = true)
 |-- Total_Value: double (nullable = true)

Sample Records:
+--------+----------+--------+----+--------+------+-----------------+
|Trade_ID|      Date|Stock_ID|Type|Quantity| Price|      Total_Value|
+--------+----------+--------+----+--------+------+-----------------+
|      T1|2024-01-01|     TCS|Sell|     347|253.46|87950.62000000001|
|      T2|2024-01-01|     TCS|Sell|     279|258.78|         72199.62|
|      T3|2024-01-01|     TCS|Sell|      27|258.55|          6980.85|
+--------+----------+--------+----+--------+------+-----------------+
only showing top 3 rows


In [27]:

print("FINANCIAL NEWS SCHEMA")
df_news.printSchema()
print("Sample Records:")
df_news.show(3)

FINANCIAL NEWS SCHEMA
root
 |-- News_ID: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- Stock_ID: string (nullable = true)
 |-- Headline: string (nullable = true)
 |-- Sentiment: string (nullable = true)
 |-- Sentiment_Score: double (nullable = true)

Sample Records:
+-------+----------+--------+------------------+---------+---------------+
|News_ID|      Date|Stock_ID|          Headline|Sentiment|Sentiment_Score|
+-------+----------+--------+------------------+---------+---------------+
|     N1|2024-01-01|     TCS|       New project| Positive|           0.89|
|     N2|2024-01-01|    INFY|Market uncertainty|  Neutral|          -0.03|
|     N3|2024-01-01|RELIANCE|     Strong demand| Negative|          -0.53|
+-------+----------+--------+------------------+---------+---------------+
only showing top 3 rows


Q2. RDD Implementation

In [28]:
# Step 1: Convert our DataFrame into a raw RDD
trading_rdd = df_trading.rdd
print("Data successfully converted to RDD format!")

Data successfully converted to RDD format!


In [31]:
# Step 2: Apply a Transformation (Filter)
# Let's say we only want to look at transactions where the trading Volume is greater than 10,000 shares.
# (Note: Spark hasn't actually done the filtering yet; it's just planning it!)
# We change 'Volume' to 'Quantity' to match your actual schema!
high_volume_rdd = trading_rdd.filter(lambda row: row['Quantity'] is not None and row['Quantity'] > 100)

In [32]:
# Step 3: Apply an Action (Count)
total_high_volume_records = high_volume_rdd.count()

print(f"\n[Action Result] Total rows with Quantity > 100: {total_high_volume_records}")


[Action Result] Total rows with Quantity > 100: 1218


In [34]:
# Step 4: Apply another Action (Take)
# Let's grab just the first 3 records of our filtered RDD to look at them.
print("\n[Action Result] Displaying 3 high-volume rows:")
for record in high_volume_rdd.take(3):
    print(record)


[Action Result] Displaying 3 high-volume rows:
Row(Trade_ID='T1', Date=datetime.date(2024, 1, 1), Stock_ID='TCS', Type='Sell', Quantity=347, Price=253.46, Total_Value=87950.62000000001)
Row(Trade_ID='T2', Date=datetime.date(2024, 1, 1), Stock_ID='TCS', Type='Sell', Quantity=279, Price=258.78, Total_Value=72199.62)
Row(Trade_ID='T4', Date=datetime.date(2024, 1, 1), Stock_ID='INFY', Type='Sell', Quantity=217, Price=242.99, Total_Value=52728.83)


Q3. Key-Value Operations and Persistence

In [35]:
from pyspark.storagelevel import StorageLevel

# Step 1: Create a Key-Value Pair RDD -> (Stock_ID, Quantity)
# This maps each row to a simple tuple
stock_pairs_rdd = trading_rdd.map(lambda row: (row['Stock_ID'], int(row['Quantity'])))

In [36]:
# Step 2: Shuffle & Aggregate (reduceByKey)
# This finds all matching Stock_IDs and sums up their quantities
total_stock_volumes = stock_pairs_rdd.reduceByKey(lambda current_sum, next_value: current_sum + next_value)

In [37]:
# Step 3: Persistence
# This tells Spark to keep this result cached in memory so future actions are instant
total_stock_volumes.persist(StorageLevel.MEMORY_AND_DISK)

PythonRDD[156] at RDD at PythonRDD.scala:56

In [38]:
# Step 4: Run an action to see our aggregated results
print("Total Trading Quantities per Stock (Key-Value Result):")
for stock, total_qty in total_stock_volumes.collect():
    print(f"Stock ID: {stock} | Total Quantity Traded: {total_qty}")

Total Trading Quantities per Stock (Key-Value Result):
Stock ID: TCS | Total Quantity Traded: 76214
Stock ID: INFY | Total Quantity Traded: 73001
Stock ID: RELIANCE | Total Quantity Traded: 80356
Stock ID: HDFCBANK | Total Quantity Traded: 76650
Stock ID: TATAMOTORS | Total Quantity Traded: 73022


Q4. Spark DataFrame Operations

In [39]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("--- DataFrame Filtering & Selection ---")
# 1. Filter out only the rows where the stock belong to the 'IT' sector
it_stocks = df_stock.filter(df_stock["Sector"] == "IT")
it_stocks.select("Date", "Stock_ID", "Close", "Sentiment_Score").show(3)

--- DataFrame Filtering & Selection ---
+----------+--------+------+---------------+
|      Date|Stock_ID| Close|Sentiment_Score|
+----------+--------+------+---------------+
|2024-01-01|     TCS|255.41|           0.89|
|2024-01-01|    INFY|250.11|          -0.03|
|2024-01-02|     TCS|240.69|          -0.72|
+----------+--------+------+---------------+
only showing top 3 rows


In [40]:
print("\n--- Grouping & Aggregations ---")
# 2. Find the Average Closing Price and Maximum Sentiment Score for each Sector
sector_summary = df_stock.groupBy("Sector").agg(
    F.avg("Close").alias("Average_Close"),
    F.max("Sentiment_Score").alias("Max_Sentiment")
)
sector_summary.show()


--- Grouping & Aggregations ---
+-------+------------------+-------------+
| Sector|     Average_Close|Max_Sentiment|
+-------+------------------+-------------+
| Energy|294.33040000000017|         0.98|
|     IT|          299.5627|          1.0|
|Banking| 284.8927000000001|         0.98|
|   Auto| 299.5953000000001|         0.97|
+-------+------------------+-------------+



In [41]:
print("\n--- Window Function (Moving Average) ---")
# 3. Calculate a rolling 2-day average closing price for each Stock_ID ordered by Date
window_spec = Window.partitionBy("Stock_ID").orderBy("Date").rowsBetween(-1, 0)

df_with_moving_avg = df_stock.withColumn("2_Day_Moving_Avg", F.avg("Close").over(window_spec))
df_with_moving_avg.select("Date", "Stock_ID", "Close", "2_Day_Moving_Avg").show(6)


--- Window Function (Moving Average) ---
+----------+--------+------+------------------+
|      Date|Stock_ID| Close|  2_Day_Moving_Avg|
+----------+--------+------+------------------+
|2024-01-01|HDFCBANK|141.39|            141.39|
|2024-01-02|HDFCBANK|150.52|145.95499999999998|
|2024-01-03|HDFCBANK|152.97|           151.745|
|2024-01-04|HDFCBANK|133.92|           143.445|
|2024-01-05|HDFCBANK|358.02|245.96999999999997|
|2024-01-06|HDFCBANK|284.78|             321.4|
+----------+--------+------+------------------+
only showing top 6 rows


Q5. Exploratory Data Analysis (EDA) and Spark SQL

In [42]:
# First, let's create SQL views out of your DataFrames
df_stock.createOrReplaceTempView("stock_market")
df_trading.createOrReplaceTempView("historical_trading")

In [43]:
print("--- 1. Top Performing Stocks (Based on Closing Prices) ---")
spark.sql("""
    SELECT Stock_ID, AVG(Close) as Avg_Close_Price
    FROM stock_market
    GROUP BY Stock_ID
    ORDER BY Avg_Close_Price DESC
""").show(5)

--- 1. Top Performing Stocks (Based on Closing Prices) ---
+----------+------------------+
|  Stock_ID|   Avg_Close_Price|
+----------+------------------+
|      INFY|312.70980000000014|
|TATAMOTORS| 299.5953000000001|
|  RELIANCE|294.33040000000017|
|       TCS| 286.4155999999999|
|  HDFCBANK| 284.8927000000001|
+----------+------------------+



In [44]:
print("\n--- 2. Total Trading Quantity per Day ---")
spark.sql("""
    SELECT Date, SUM(Quantity) as Total_Daily_Quantity
    FROM historical_trading
    GROUP BY Date
    ORDER BY Date
""").show(5)


--- 2. Total Trading Quantity per Day ---
+----------+--------------------+
|      Date|Total_Daily_Quantity|
+----------+--------------------+
|2024-01-01|                4726|
|2024-01-02|                4524|
|2024-01-03|                4649|
|2024-01-04|                3944|
|2024-01-05|                3915|
+----------+--------------------+
only showing top 5 rows


In [45]:
print("\n--- 3. Sector-wise Summary ---")
spark.sql("""
    SELECT Sector, COUNT(Distinct Stock_ID) as Total_Stocks, AVG(Close) as Avg_Price
    FROM stock_market
    GROUP BY Sector
""").show()


--- 3. Sector-wise Summary ---
+-------+------------+------------------+
| Sector|Total_Stocks|         Avg_Price|
+-------+------------+------------------+
| Energy|           1|294.33040000000017|
|     IT|           2|299.56270000000006|
|Banking|           1| 284.8927000000001|
|   Auto|           1| 299.5953000000001|
+-------+------------+------------------+



In [46]:
print("\n--- 4. Unusual Trading Activities (Large Orders > 450 Quantity) ---")
spark.sql("""
    SELECT Trade_ID, Date, Stock_ID, Type, Quantity, Price
    FROM historical_trading
    WHERE Quantity > 450
    ORDER BY Quantity DESC
""").show(5)


--- 4. Unusual Trading Activities (Large Orders > 450 Quantity) ---
+--------+----------+--------+----+--------+------+
|Trade_ID|      Date|Stock_ID|Type|Quantity| Price|
+--------+----------+--------+----+--------+------+
|    T232|2024-01-16|RELIANCE|Sell|     500|252.39|
|    T612|2024-02-10|HDFCBANK|Sell|     500|211.71|
|    T746|2024-02-19|HDFCBANK|Sell|     500|280.59|
|    T213|2024-01-15|     TCS| Buy|     499|282.37|
|    T663|2024-02-14|     TCS| Buy|     499|329.42|
+--------+----------+--------+----+--------+------+
only showing top 5 rows


In [47]:
print("\n--- 5. Stock Trend Report (Simple Moving Average) ---")
spark.sql("""
    SELECT Date, Stock_ID, Close,
           AVG(Close) OVER(PARTITION BY Stock_ID ORDER BY Date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as 3_Day_Moving_Avg
    FROM stock_market
    ORDER BY Stock_ID, Date
""").show(6)


--- 5. Stock Trend Report (Simple Moving Average) ---
+----------+--------+------+------------------+
|      Date|Stock_ID| Close|  3_Day_Moving_Avg|
+----------+--------+------+------------------+
|2024-01-01|HDFCBANK|141.39|            141.39|
|2024-01-02|HDFCBANK|150.52|145.95499999999998|
|2024-01-03|HDFCBANK|152.97|148.29333333333332|
|2024-01-04|HDFCBANK|133.92| 145.8033333333333|
|2024-01-05|HDFCBANK|358.02|            214.97|
|2024-01-06|HDFCBANK|284.78|258.90666666666664|
+----------+--------+------+------------------+
only showing top 6 rows


Q6. ETL Pipeline Development

In [48]:
def run_simple_etl():
    print("Starting ETL Pipeline...")

    # 1. EXTRACT: Read data from your files
    raw_stock = spark.read.csv("/content/stock_market_data.csv", header=True, inferSchema=True)

    # 2. TRANSFORM: Clean missing rows and add a new column (High minus Low price range)
    cleaned_stock = raw_stock.dropna(subset=["Stock_ID", "Close"])
    transformed_stock = cleaned_stock.withColumn("Daily_Price_Range", F.col("High") - F.col("Low"))

    # 3. LOAD: Save the processed data into a folder as an optimized Parquet file
    output_directory = "/content/financial_analytics_warehouse"
    transformed_stock.write.mode("overwrite").parquet(output_directory)

    print(f"ETL Complete! Cleaned files saved to folder: {output_directory}")
    return transformed_stock

In [49]:
# Run the pipeline and keep the output for Machine Learning
ml_ready_df = run_simple_etl()

Starting ETL Pipeline...
ETL Complete! Cleaned files saved to folder: /content/financial_analytics_warehouse


Q7. Machine Learning Implementation

In [50]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

print("--- Machine Learning Pipeline ---")

--- Machine Learning Pipeline ---


In [51]:
# Step 1: Put features into a single vector column that Spark ML understands
feature_columns = ["Open", "High", "Low"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
ml_data = assembler.transform(ml_ready_df).select("features", F.col("Close").alias("label"))

In [52]:
# Step 2: Split data into Training (80%) and Testing (20%) sets
train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)

In [53]:
# Step 3: Initialize and train a Linear Regression model
lr = LinearRegression(featuresCol="features", labelCol="label")
model = lr.fit(train_data)
print("Model training successfully completed!")

Model training successfully completed!


In [54]:
# Step 4: Make predictions on our test data
predictions = model.transform(test_data)
print("\nSample Predictions vs Actual Values:")
predictions.select("label", F.round("prediction", 2).alias("Predicted_Close")).show(5)


Sample Predictions vs Actual Values:
+------+---------------+
| label|Predicted_Close|
+------+---------------+
| 98.73|         100.99|
|106.65|         107.45|
|109.03|         109.81|
|115.53|          117.1|
|111.98|         109.29|
+------+---------------+
only showing top 5 rows


In [55]:
# Step 5: Evaluate how well our model did using R-squared (higher is better)
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
r2_score = evaluator.evaluate(predictions)
print(f"\nModel Accuracy (R-squared Score): {r2_score:.4f}")


Model Accuracy (R-squared Score): 0.9998
